# Analyze the oracle comparison

Read `oracle_results_fixed_r.csv` from this notebook's directory. [02_global_oracle.ipynb](02_global_oracle.ipynb) generates this file from the activation archive.

Each row records a method's selected mask at a requested budget $R$. `gap_upper_pct` bounds the importance/latency ratio loss relative to the optimal mask with exactly $R$ rows. Paper may select fewer rows; those cases have no fixed-$R$ gap. Empty selections and zero-importance cases also have no ratio gap.

The figures use the saved importance, modeled latency and run intervals.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path("oracle_results_fixed_r.csv")

In [ ]:
results = pd.read_csv(RESULTS)
results.head()

## 1. Compare ratio gaps and fragmentation

In [ ]:
frame = results[results.status == "ok"]
eligible = frame[(frame.R_selected == frame.R_nominal) & frame.gap_upper_pct.notna()]

In [ ]:
gaps = eligible[eligible.method != "Global DP"].groupby(["shape", "method"]).gap_upper_pct.median().unstack()
ax = gaps.plot.bar(figsize=(8, 3.5), rot=0)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
plt.ylim(0, 105)
plt.xlabel("Projection shape")
plt.ylabel("Median ratio-gap upper bound (%)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 1.** Median ratio-gap upper bounds by shape. The table gives case counts, minimum fill percentage, median gap, run count and short-run overhead. Paper's gap uses only exact-fill cases with a defined ratio, so its bars and the other methods' bars may describe different subsets. Missing gaps have no eligible cases.

The short-run deficit is $D(M)=\sum_C(s-|C|)_+$, where $s$ is the fitted boundary in rows. With short and long slopes $b_1,b_2$, the saved overhead is $(b_2-b_1)D(M)$ and total cost is $L=b_2|M|+(b_2-b_1)D(M)$.

In [ ]:
summary = results.groupby(["shape", "method"]).agg(
    cases=("method", "size"), gap_cases=("gap_upper_pct", "count"),
    min_fill_pct=("fill_pct", "min"), median_gap_pct=("gap_upper_pct", "median"),
    median_runs=("chunks", "median"), median_overhead_ms=("overhead_ms", "median"))
summary.round(3)

## 2. Inspect importance–latency trajectories

In [ ]:
for shape, data in frame.groupby("shape", sort=False):
    points = data[data.trace_id == data.trace_id.min()]
    plt.figure(figsize=(8, 3.5))
    for name, group in points.groupby("method", sort=False):
        group = group.sort_values("fraction")
        plt.plot(group.latency, group.retained_pct, "o-", label=name, ms=4)
        if name == "Global DP":
            for row in group.itertuples():
                plt.annotate(str(row.R_nominal), (row.latency, row.retained_pct), fontsize=7)
    plt.xscale("log")
    plt.xlabel("Model latency L (ms; log scale)")
    plt.ylabel("Retained importance (%)")
    plt.title(f"{shape}: trace {points.trace_id.iloc[0]}; labels are requested R")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

**Figure 2.** Importance–latency trajectories for the first available trace of each shape. Points are masks; lines connect requested budgets, and oracle labels give $R$. Paper may underfill. Latency can decrease as intervals merge and their short-run penalty falls. Empty and zero-importance cases are omitted. Fixed-$R$ ratio optima need not form the complete Pareto frontier.

## 3. Inspect selected runs

In [ ]:
def saved_mask(row):
    mask = np.zeros(int(row.N), bool)
    for start, end in json.loads(row.intervals):
        mask[start:end] = True
    return mask


first = results[results.trace_id == results.trace_id.min()]
fraction = first.fraction.iloc[(first.fraction - .5).abs().argmin()]
example = first[np.isclose(first.fraction, fraction)].set_index("method")
methods = {name: saved_mask(row) for name, row in example.iterrows()}

In [ ]:
example[["R_nominal", "R_selected", "importance", "latency",
         "chunks", "gap_upper_pct"]].round(3)

In [ ]:
N, R = int(example.N.iloc[0]), int(example.R_nominal.iloc[0])
plt.figure(figsize=(9, 2.4))
plt.imshow(np.array(list(methods.values())), aspect="auto", interpolation="nearest")
plt.yticks(range(len(methods)), [f"{name} (selected={mask.sum()})" for name, mask in methods.items()])
plt.xticks(np.linspace(0, N - 1, 8, dtype=int))
plt.xlabel(f"Channel (trace {int(example.trace_id.iloc[0])}; requested R = {R})")
plt.tight_layout()
plt.show()

**Figure 3.** Bright cells are selected channels. The example uses the first available trace and its available budget closest to one half. The table gives importance and cost; the image shows how selection is distributed across runs. Short runs incur the deficit penalty above, but a valuable short run can still improve the ratio.

## 4. Summarize the comparison

The tables below summarize gaps over all eligible cases and then compare Paper and tiles on the same cases, matched by trace, shape and requested budget. Medians weight each case equally. A missing median means that no eligible cases exist.

In [ ]:
gap_table = eligible.pivot(index=["trace_id", "shape", "R_nominal", "fraction"],
                           columns="method", values="gap_upper_pct").reindex(columns=results.method.unique())
paired = gap_table.reindex(columns=["Paper greedy", "Saturation tiles"]).dropna()
pd.concat({"All eligible cases": gap_table.agg(["median", "count"]).T,
           "Matched Paper / tiles": paired.agg(["median", "count"]).T},
          names=["Comparison", "Method"]).rename(
              columns={"median": "Median gap upper bound (%)", "count": "Cases"}).round(3)

The oracle searches the full fixed-$R$ mask space to the specified ratio tolerance. Its gaps quantify how much ratio each heuristic leaves under the fitted cost. The decomposition $L=b_2|M|+(b_2-b_1)D(M)$ connects that comparison to run geometry: at equal row count, differences in modeled latency come entirely from short-run deficit.

The conclusions concern activation-magnitude importance and an additive latency model. They do not establish end-to-end accuracy or measured SSD speedup; native task splitting and scheduling are outside this model.